In [12]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

GMINI_API_KEY = os.getenv("GIMINI_API_KEY")
def get_llm(completion: int = 1):
    temperature = 0
    if completion > 1:
        temperature = 0.2
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=temperature, google_api_key=GMINI_API_KEY)
    return llm
from utils import voting

In [7]:
from langchain.prompts import ChatPromptTemplate

from prompts import HUMAN_PROMPT, SELECTION_SYSTEM_PROMPT
from schemas import ContextualSentence, SelectedContent, SelectionOutput
llm = get_llm(1)

In [8]:
async def single_selection_attempt(context: ContextualSentence, llm_instance):
    messages = ChatPromptTemplate(
        [
            ("system", SELECTION_SYSTEM_PROMPT),
            ("human", HUMAN_PROMPT),
        ]
    )

    prompt_messages = messages.invoke(
        {
            "excerpt": context.context,
            "sentence": context.sentence,
        }
    )

    response = llm_instance.with_structured_output(SelectionOutput).invoke(prompt_messages)
    
    if (
        not response
        or not response.processed_sentence
        or response.no_verifiable_claims
    ):
        return False, None
    if response.remains_unchanged:
        processed = context.sentence
    else:
        processed = response.processed_sentence.strip()
    return True, processed
        
def create_selected_content(
    processed_sentence: str,
    context: ContextualSentence,
) -> SelectedContent:
    return SelectedContent(
        processed_sentence=processed_sentence,
        original_context_item=context,
    )

In [9]:
import asyncio

from loguru import logger
from splitter import sentence_splitter
from typing import Any, Callable, List, Optional, Tuple, TypeVar
T = TypeVar("T")
R = TypeVar("R")
text = """A group of flamingos is called a "flamboyance," and surprisingly, honey never spoils, with archaeologists even finding edible pots of it in ancient Egyptian tombs over 3,000 years old. The shortest war in history lasted only 38 minutes between Britain and Zanzibar in 1896, while biologically, octopuses have three hearts. Many people find it impossible to lick their own elbow, and contrary to popular belief, the Great Wall of China is not visible from space with the naked eye. Interestingly, a strawberry isn't technically a berry (but a banana is), and the human nose can remember 50,000 different scents, while butterflies taste with their feet."""

sentences = sentence_splitter(text)

# Properly run the async voting function
result = await voting(
    items=sentences,
    single_attempt_function=single_selection_attempt,
    llm=llm,
    completions=1,
    min_successes=1,
    result_factory=create_selected_content,
    description="Selecting sentences from a text that are verifiable claims."    
)

In [10]:
# print result
print(result)

[SelectedContent(processed_sentence='A group of flamingos is called a "flamboyance," and surprisingly, honey never spoils, with archaeologists even finding edible pots of it in ancient Egyptian tombs over 3,000 years old.', original_context_item=ContextualSentence(sentence='A group of flamingos is called a "flamboyance," and surprisingly, honey never spoils, with archaeologists even finding edible pots of it in ancient Egyptian tombs over 3,000 years old.', context='\n[Sentence of Interest for current task:]\nA group of flamingos is called a "flamboyance," and surprisingly, honey never spoils, with archaeologists even finding edible pots of it in ancient Egyptian tombs over 3,000 years old.\n\n[Following Sentences:]\nThe shortest war in history lasted only 38 minutes between Britain and Zanzibar in 1896, while biologically, octopuses have three hearts.\nMany people find it impossible to lick their own elbow, and contrary to popular belief, the Great Wall of China is not visible from sp